In [1]:
import pandas as pd
import requests
import time
import csv
from pathlib import Path
from datetime import datetime
import trafilatura

# Load the URL extraction we did earlier
# GDELT_URLS_CSV = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01\gdelt_articles.csv")
GDELT_URLS_CSV = Path(r"D:\Shikha\bias_project\corpus\gdelt\2015-03\gdelt_articles.csv")
df = pd.read_csv(
    GDELT_URLS_CSV,
    usecols=['date', 'source_domain', 'article_url', 'state', 'source_name', 'source_size']
)


print(f"Loaded {len(df):,} article URLs from GDELT extraction")
print(f"States covered: {df['state'].nunique()}")

# Stratified sampling: ~30 articles per state
SAMPLES_PER_STATE = 30
TARGET_TOTAL = SAMPLES_PER_STATE * df['state'].nunique()

print(f"\nTarget: {SAMPLES_PER_STATE} articles per state = ~{TARGET_TOTAL} total")

# Sample stratified by state
sampled_dfs = []
for state in df['state'].unique():
    state_articles = df[df['state'] == state]
    n = min(SAMPLES_PER_STATE, len(state_articles))
    sampled_dfs.append(state_articles.sample(n=n, random_state=42))

sample_df = pd.concat(sampled_dfs, ignore_index=True)
# Shuffle so we don't fetch all of one state in a row (be polite to sites)
sample_df = sample_df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Actual sample size: {len(sample_df):,} articles")
print(f"\nArticles per state in sample:")
print(sample_df['state'].value_counts().sort_index())

Loaded 358,715 article URLs from GDELT extraction
States covered: 50

Target: 30 articles per state = ~1500 total
Actual sample size: 1,500 articles

Articles per state in sample:
state
AK    30
AL    30
AR    30
AZ    30
CA    30
CO    30
CT    30
DC    30
DE    30
FL    30
GA    30
HI    30
IA    30
ID    30
IL    30
IN    30
KS    30
KY    30
LA    30
MA    30
MD    30
ME    30
MI    30
MN    30
MO    30
MS    30
MT    30
NC    30
ND    30
NE    30
NH    30
NJ    30
NM    30
NV    30
NY    30
OH    30
OK    30
OR    30
PA    30
RI    30
SC    30
SD    30
TN    30
TX    30
UT    30
VA    30
VT    30
WA    30
WI    30
WY    30
Name: count, dtype: int64


In [2]:
# Fetch article text for each URL in the sample
# Uses trafilatura for clean extraction, with retries and resume support

import csv

# OUTPUT_DIR = Path(r"D:\Shikha\bias_project\corpus\gdelt\2024-01")
OUTPUT_DIR = Path(r"D:\Shikha\bias_project\corpus\gdelt\2015-03")
TEXT_CSV = OUTPUT_DIR / "gdelt_articles_with_text.csv"
print("WILL WRITE TO:", TEXT_CSV)
ERRORS_LOG = OUTPUT_DIR / "fetch_errors.log"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Realistic browser user-agent (avoids basic bot detection)
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                  "(KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

REQUEST_TIMEOUT = 15
FETCH_DELAY = 1.5  # be polite

# Resume support — check what's already done
processed_urls = set()
if TEXT_CSV.exists():
    with open(TEXT_CSV, 'r', encoding='utf-8', newline='') as f:
        reader = csv.DictReader(f)
        for row in reader:
            processed_urls.add(row['article_url'])
    print(f"Resume mode: {len(processed_urls)} URLs already processed, skipping those")

# Columns for output CSV
TEXT_FIELDS = [
    'article_url', 'state', 'source_domain', 'source_name', 'source_size',
    'article_date', 'fetch_status', 'text_length', 'text'
]

# Open CSV in append mode
file_exists = TEXT_CSV.exists()
csv_file = open(TEXT_CSV, 'a', encoding='utf-8', newline='')
writer = csv.DictWriter(csv_file, fieldnames=TEXT_FIELDS)
if not file_exists:
    writer.writeheader()

errors_file = open(ERRORS_LOG, 'a', encoding='utf-8')

stats = {
    'attempted': 0,
    'skipped_already_done': 0,
    'success': 0,
    'paywall_stub': 0,
    'fetch_failed': 0,
    'no_text_extracted': 0,
}

start_time = time.time()
total = len(sample_df)

print(f"\nFetching text for {total:,} articles...")
print(f"Output: {TEXT_CSV}\n")

for i, row in sample_df.iterrows():
    url = row['article_url']
    
    # Skip if already processed (resume support)
    if url in processed_urls:
        stats['skipped_already_done'] += 1
        continue
    
    stats['attempted'] += 1
    
    # Build output row template
    output_row = {
        'article_url': url,
        'state': row['state'],
        'source_domain': row['source_domain'],
        'source_name': row['source_name'],
        'source_size': row['source_size'],
        'article_date': row.get('article_date', ''),
        'fetch_status': '',
        'text_length': 0,
        'text': '',
    }
    
    # Polite delay
    time.sleep(FETCH_DELAY)
    
    # Fetch
    try:
        resp = requests.get(url, headers=HEADERS, timeout=REQUEST_TIMEOUT, allow_redirects=True)
        
        if resp.status_code != 200:
            output_row['fetch_status'] = f"http_{resp.status_code}"
            stats['fetch_failed'] += 1
            errors_file.write(f"{url}\thttp_{resp.status_code}\n")
            writer.writerow(output_row)
            csv_file.flush()
            continue
        
        # Extract clean text with trafilatura
        text = trafilatura.extract(resp.text, include_comments=False, include_tables=False)
        
        if not text:
            output_row['fetch_status'] = 'no_text_extracted'
            stats['no_text_extracted'] += 1
            errors_file.write(f"{url}\tno_text_extracted\n")
            writer.writerow(output_row)
            csv_file.flush()
            continue
        
        # Heuristic paywall detection: very short text often means paywall stub
        if len(text) < 200:
            output_row['fetch_status'] = 'paywall_stub_suspected'
            output_row['text_length'] = len(text)
            output_row['text'] = text
            stats['paywall_stub'] += 1
            writer.writerow(output_row)
            csv_file.flush()
            continue
        
        # Success
        output_row['fetch_status'] = 'success'
        output_row['text_length'] = len(text)
        output_row['text'] = text
        stats['success'] += 1
        writer.writerow(output_row)
        csv_file.flush()
        
    except requests.exceptions.Timeout:
        output_row['fetch_status'] = 'timeout'
        stats['fetch_failed'] += 1
        errors_file.write(f"{url}\ttimeout\n")
        writer.writerow(output_row)
        csv_file.flush()
    
    except requests.exceptions.RequestException as e:
        output_row['fetch_status'] = f'error_{type(e).__name__}'
        stats['fetch_failed'] += 1
        errors_file.write(f"{url}\t{type(e).__name__}\t{str(e)[:100]}\n")
        writer.writerow(output_row)
        csv_file.flush()
    
    except Exception as e:
        output_row['fetch_status'] = f'unexpected_{type(e).__name__}'
        stats['fetch_failed'] += 1
        errors_file.write(f"{url}\tunexpected\t{type(e).__name__}\t{str(e)[:100]}\n")
        writer.writerow(output_row)
        csv_file.flush()
    
    # Progress update every 50 fetches
    if (i + 1) % 50 == 0:
        elapsed_min = (time.time() - start_time) / 60
        rate = (i + 1) / elapsed_min if elapsed_min > 0 else 0
        eta_min = (total - i - 1) / rate if rate > 0 else 0
        print(f"  [{i+1}/{total}] elapsed {elapsed_min:.1f}m, "
              f"rate {rate:.0f}/min, ETA {eta_min:.0f}m | "
              f"success {stats['success']}, "
              f"paywall {stats['paywall_stub']}, "
              f"failed {stats['fetch_failed']}, "
              f"no_text {stats['no_text_extracted']}")

csv_file.close()
errors_file.close()

elapsed_min = (time.time() - start_time) / 60

print(f"\n--- Fetching Complete ---")
print(f"Total time:              {elapsed_min:.1f} minutes")
print(f"Attempted:               {stats['attempted']:,}")
print(f"Already done (skipped):  {stats['skipped_already_done']:,}")
print(f"Successful (real text):  {stats['success']:,}")
print(f"Paywall stubs (short):   {stats['paywall_stub']:,}")
print(f"No text extracted:       {stats['no_text_extracted']:,}")
print(f"Fetch failed:            {stats['fetch_failed']:,}")
print(f"\nOutput written to:       {TEXT_CSV}")
print(f"Errors logged to:        {ERRORS_LOG}")

WILL WRITE TO: D:\Shikha\bias_project\corpus\gdelt\2015-03\gdelt_articles_with_text.csv

Fetching text for 1,500 articles...
Output: D:\Shikha\bias_project\corpus\gdelt\2015-03\gdelt_articles_with_text.csv

  [50/1500] elapsed 4.3m, rate 12/min, ETA 126m | success 20, paywall 0, failed 30, no_text 0
  [300/1500] elapsed 20.2m, rate 15/min, ETA 81m | success 104, paywall 2, failed 193, no_text 1
  [350/1500] elapsed 24.6m, rate 14/min, ETA 81m | success 119, paywall 2, failed 228, no_text 1
  [450/1500] elapsed 31.7m, rate 14/min, ETA 74m | success 164, paywall 2, failed 283, no_text 1
  [500/1500] elapsed 33.5m, rate 15/min, ETA 67m | success 186, paywall 2, failed 311, no_text 1
  [550/1500] elapsed 36.3m, rate 15/min, ETA 63m | success 212, paywall 2, failed 335, no_text 1
  [700/1500] elapsed 47.2m, rate 15/min, ETA 54m | success 261, paywall 3, failed 435, no_text 1
  [750/1500] elapsed 50.2m, rate 15/min, ETA 50m | success 285, paywall 3, failed 461, no_text 1
  [900/1500] elapsed